In [0]:
# 0. SETUP E IMPORTS
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.getcwd())
sys.modules["flash_attn"] = None
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import pandas as pd
from chronos import Chronos2Pipeline
import numpy as np
import mlflow
import pyspark.sql.functions as F
from datetime import date, timedelta, datetime
from mlflow.tracking import MlflowClient

# Módulos do projeto
from src.validation.config import Config
from src.validation.data import DataIngestion

# Configs Spark
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")
import warnings

# SetUp
warnings.filterwarnings("ignore")
client = MlflowClient()
print("✅ Setup concluído.")

In [0]:
# 1. CONFIGURAÇÃO DA JANELA DE CONTEXTO
# 'Hoje' — em produção, usar date.today()
today = date.today()

# Janela de lookback: suficiente para os lags do modelo (max_lag=15) + margem
# O wrapper usa max_lag=15 por padrão. Usamos 90 dias para ter histórico farto.
context_days = 12*3
start_context = today - timedelta(days=context_days)

# Config dinâmica (mesmos parâmetros do treino)
config = Config(spark)
config.DATA_START  = start_context.strftime("%Y-%m-%d")
config.INGESTION_END = today.strftime("%Y-%m-%d")
config.SCHEMA = "cvc_val"
# Horizonte de previsão (em dias)
FORECAST_HORIZON = config.FORECAST_HORIZON

print(f"📅 Data de Referência (Hoje): {today}")
print(f"🔎 Janela de contexto: {config.DATA_START} → {config.INGESTION_END}")
print(f"🔮 Horizonte de previsão: {FORECAST_HORIZON} dias")

In [0]:
# 2. CARREGAMENTO DO HISTÓRICO VIA FEATURE STORE
#    Usa exatamente o mesmo DataIngestion do treino — garante consistência
#    nas features estáticas (cluster_loja, sigla_uf, tipo_loja, modelo_loja)
#    e covariáveis locais (is_feriado).

print("⏳ Carregando histórico recente via Feature Store...")
ingestion = DataIngestion(spark, config)
df_spark_raw = ingestion.create_training_set()

# Filtro de segurança (a DataIngestion já filtra, mas garantimos aqui)
df_spark_raw = df_spark_raw.filter(
    F.col("data").between(config.DATA_START, config.INGESTION_END)
)

# Converte para Pandas — volume pequeno (janela de 90 dias)
df_context_pd = df_spark_raw.toPandas()
df_context_pd['data'] = pd.to_datetime(df_context_pd['data'])
df_context_pd['codigo_loja'] = (
    df_context_pd['codigo_loja']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
)

print(f"✅ Histórico carregado: {len(df_context_pd)} linhas | {df_context_pd['codigo_loja'].nunique()} lojas")
print(f"   Colunas: {df_context_pd.columns.tolist()}")

In [0]:
# 3. CARREGAMENTO DOS INDICADORES DE MERCADO (historico_suporte_loja)
#    Segue o mesmo padrão de get_global_support() do data.py:
#    - Sem filtro de data (carrega tudo para ter cobertura futura)
#    - Pivot por 'metricas'
#    - Frequência diária contínua (asfreq + ffill)
#    - Extensão para cobrir o FORECAST_HORIZON

print("📊 Carregando indicadores de mercado (suporte global)...")
df_market_spark = (
    spark.table(f"{config.CATALOG}.{config.SCHEMA}.historico_suporte_venda_direta")
    .groupBy("data")
    .pivot("metricas")
    .agg(F.sum("valor"))
    .na.fill(0.0)
)

pdf_market = df_market_spark.toPandas()
pdf_market['data'] = pd.to_datetime(pdf_market['data'])

# Garante frequência diária contínua (igual ao data.py)
pdf_market = (
    pdf_market
    .set_index('data')
    .asfreq('D')
    .fillna(0.0)
)

# Extensão futura: cobre o horizonte de previsão (igual ao get_global_support)
full_market_range = pd.date_range(
    start=pdf_market.index.min(),
    # Aumente a margem de 15 para 60 ou mais para garantir cobertura
    periods=len(pdf_market) + FORECAST_HORIZON + 60, 
    freq='D'
)
pdf_market = pdf_market.reindex(full_market_range).ffill().fillna(0.0).reset_index()
pdf_market.rename(columns={'index': 'data'}, inplace=True)
pdf_market.drop(columns=[ 'BM12_PIB12', 'CNC12_ICF12', 'CNC12_ICFAB12', 'CNC12_ICFAC12',
       'CNC12_ICFAJ12', 'GAC12_PPCTAXAC12', 'IPP12_IPPC10ATIV12',
       'PMC12_VRSUPN12', 'PMC12_VRSUPNSA12', 'PNADC12_OCUPALOJ12',
       'PNADC12_TDESOC12', 'SGS12_IBCBR12', 'SGS12_IBCBRDESSAZ12'], inplace=True)
                          
pdf_market['data'] = pd.to_datetime(pdf_market['data'])
market_cols = [c for c in pdf_market.columns if c != 'data']
print(f"✅ Mercado carregado: {len(pdf_market)} dias | Indicadores: {market_cols}")

In [0]:
df_vendas=spark.sql('''
select data, sum(valor) as valor
 from ds_hml.cvc.historico_targuet_venda_direta
 group by data
 order by data
 ''').toPandas()
df_vendas['data']=pd.to_datetime(df_vendas['data'])
date_max = max(df_vendas['data'])

In [0]:
df=pdf_market.merge(df_vendas, on='data', how='left')

Modelagem

In [0]:
df_context = df.query(f"data <= '{date_max}' and data >= '01-01-2023'")
df_future = df.query(f"data > '{date_max}'").drop(columns=['valor'])

pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

df_context['n'] = '1'
df_future['n'] = '1'

df_future = (
    df_future
    .sort_values("data")
    .groupby("n")
    .head(35))

# Generate predictions with covariates
pred_df = pipeline.predict_df(
    df_context,
    future_df = df_future,
    prediction_length = 35,  # Number of steps to forecast
    quantile_levels = [0.1, 0.9],  # Quantiles for probabilistic forecast
    id_column="n",  # Column identifying different time series
    timestamp_column="data",  # Column with datetime information
    target="valor",  # Column(s) with time series values to predict
)

In [0]:
import plotly.express as px
import plotly.graph_objects as go

fig = go.Figure()

# Linha principal (previsão)
fig.add_trace(
    go.Scatter(
        x=pred_df["data"],
        y=pred_df["predictions"],
        mode="lines",
        name="Previsão",
        line=dict(color="#1f77b4", width=2),
    )
)

# Limite inferior (0.1)
fig.add_trace(
    go.Scatter(
        x=pred_df["data"],
        y=pred_df["0.1"],
        mode="lines",
        name="P10",
        line=dict(width=0),
        showlegend=False,
    )
)

# Limite superior (0.9) + preenchimento
fig.add_trace(
    go.Scatter(
        x=pred_df["data"],
        y=pred_df["0.9"],
        mode="lines",
        name="Intervalo de Incerteza (P10–P90)",
        fill="tonexty",
        fillcolor="rgba(31, 119, 180, 0.2)",
        line=dict(width=0),
    )
)

fig.update_layout(
    title="Previsão – Chronos (com intervalo de incerteza)",
    xaxis_title="Data",
    yaxis_title="Valor",
    hovermode="x unified",
    template="plotly_white",
)

fig.show()

In [0]:
df_output = pred_df.rename(columns = {'predictions': 'forecast', '0.1': 'lower', '0.9': 'upper'}).drop(columns = ['target_name','n'])
df_output['data_reference'] = datetime.now()

In [0]:
output_table = f"{config.CATALOG}.{config.SCHEMA}.previsao_venda_direta_futuro_global"
print(f"💾 Salvando resultados em: {output_table}")
(
    spark.createDataFrame(df_output)
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(output_table)
)
spark.sql(f"OPTIMIZE {output_table}")
print("✨ Sucesso! Dados salvos e otimizados.")

In [0]:
output_table